# 01. Train MobileNetV3-Large Classifier

**Smart Recycling Coach** — COMP 365 Team Project.
Fine-tunes an ImageNet-pretrained MobileNetV3 on the three-class recycling dataset (`recycling`, `special_handling`, `trash`).

### Running on Google Colab:
1. Choose **Runtime > Change runtime type > T4 GPU**.
2. Run the first cell below to clone the project repository and install dependencies.
3. Select **Runtime > Run all** to execute the pipeline end-to-end.

In [ ]:
# Colab Environment Setup (Runs automatically if in Google Colab)
import sys, os
from pathlib import Path

if "google.colab" in sys.modules or os.environ.get("COLAB_GPU", False):
    print("🚀 Detected Google Colab environment.")
    repo_name = "insert-team-name-here"
    if not Path(repo_name).exists():
        print("Cloning repository...")
        !git clone https://github.com/rickeyjohnson/insert-team-name-here.git
    if Path(repo_name).exists():
        %cd {repo_name}
        !pip install -q -e .
        sys.path.insert(0, "src")
        print(f"✅ Working directory set to: {os.getcwd()}")
else:
    # Local laptop execution: add src to sys.path
    sys.path.insert(0, str(Path.cwd().parent / "src"))
    sys.path.insert(0, "src")

import torch
import pandas as pd
import matplotlib.pyplot as plt
from recycle_coach import config, ingest, clean, splits, baseline, metrics, dataset, train, predict

print(f"PyTorch: {torch.__version__} | Device: {train.get_device()}")
print(f"Canonical Classes: {config.CLASSES} | Seed: {config.SEED}")

## 1. Verify or Prepare Dataset Splits
If raw images or split manifests are not yet prepared locally, we run the automated ingest, cleaning, and splitting pipeline.

In [ ]:
# If splits.csv does not exist, run data ingestion & preparation
if not config.SPLITS_CSV.exists() or not config.CLEAN_CSV.exists():
    print("Preparing dataset from TACO annotations...")
    ingest.ingest_taco_dataset(cap_per_class=450)
    clean.run_cleaning()
    splits.run_splits()
    print("Dataset preparation complete!")

df_splits = pd.read_csv(config.SPLITS_CSV)
split_counts = pd.crosstab(df_splits["final_label"], df_splits["split"]).reindex(
    index=list(config.CLASSES), columns=["train", "val", "test"]
)
split_counts.loc["Total"] = split_counts.sum()
print(split_counts)

## 2. Evaluate Majority-Class Baseline
Compute the majority-class baseline benchmark on the validation split.

In [ ]:
import json
base_val_preds = baseline.run_baseline(split_name="val")
with open(config.METRICS_DIR / "baseline_val.json") as f:
    base_val_metrics = json.load(f)

print(f"Baseline Val Accuracy: {base_val_metrics['accuracy']:.4f}")
print(f"Baseline Val Macro F1: {base_val_metrics['macro_f1']:.4f}")

## 3. Two-Stage MobileNetV3 Fine-Tuning
- **Stage 1:** Freeze backbone, train classifier head (lr=1e-3, 3 epochs).
- **Stage 2:** Unfreeze backbone, full fine-tuning with class-weighted cross-entropy and Cosine Annealing (lr=1e-4, patience=5).

In [ ]:
training_results = train.train_model(
    stage1_epochs=3,
    stage2_epochs=15,
    patience=5,
    batch_size=32,
    pretrained=True
)

best_epoch = training_results['best_epoch']
best_f1 = training_results['best_macro_f1']
print(f"Best Checkpoint: Epoch {best_epoch} | Val Macro F1: {best_f1:.4f}")

## 4. Plot Training & Validation Curves

In [ ]:
h = pd.DataFrame(training_results["history"])
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(h["epoch"], h["train_loss"], marker="o", color="#1f77b4")
ax[0].set_title("Training Loss"); ax[0].set_xlabel("Epoch"); ax[0].grid(True, alpha=0.3)

ax[1].plot(h["epoch"], h["val_accuracy"], marker="o", label="Val Accuracy", color="#2ca02c")
ax[1].plot(h["epoch"], h["val_macro_f1"], marker="o", label="Val Macro F1", color="#ff7f0e")
ax[1].axhline(base_val_metrics["macro_f1"], ls="--", color="gray", label="Baseline Macro F1")
ax[1].set_title("Validation Metrics"); ax[1].set_xlabel("Epoch"); ax[1].legend(); ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "training_curves.png", dpi=150)
plt.show()

## 5. Held-Out Test Evaluation
Evaluate the trained MobileNetV3 model on the held-out test split.

In [ ]:
# Run predictions on test split
predictor = predict.load_predictor(checkpoint_path=config.CHECKPOINT_PATH)
df_test = df_splits[df_splits["split"] == "test"].reset_index(drop=True)

test_records = []
for _, row in df_test.iterrows():
    img_path = config.PROCESSED_DIR / f"{row['image_id']}.jpg"
    res = predictor.predict(img_path)
    test_records.append({
        "image_id": row["image_id"],
        "split": "test",
        "true_label": row["final_label"],
        "pred_label": res["label"],
        "confidence": res["confidence"],
        "p_recycling": res["probs"].get("recycling", 0.0),
        "p_special_handling": res["probs"].get("special_handling", 0.0),
        "p_trash": res["probs"].get("trash", 0.0),
    })

test_preds_csv = config.PREDICTIONS_DIR / "mobilenetv3_test.csv"
pd.DataFrame(test_records).to_csv(test_preds_csv, index=False)

test_eval = metrics.evaluate_predictions(
    predictions_csv=test_preds_csv,
    model_name="mobilenetv3",
    split_name="test",
    output_metrics_json=config.METRICS_DIR / "mobilenetv3_test.json",
    output_figure_png=config.FIGURES_DIR / "confusion_mobilenetv3_test.png",
    output_error_csv=config.ERRORS_DIR / "mobilenetv3_test_errors.csv",
)

print(f"Held-Out Test Accuracy: {test_eval['accuracy']:.4f}")
print(f"Held-Out Test Macro F1: {test_eval['macro_f1']:.4f}")
print(f"Baseline Test Macro F1: {base_val_metrics['macro_f1']:.4f}")

## 6. Live Prediction Demo
Test prediction on a sample photo or your own uploaded photo.

In [ ]:
sample_files = list(config.PROCESSED_DIR.glob("*.jpg"))
if sample_files:
    demo_img = sample_files[0]
    print(f"Predicting on sample: {demo_img.name}")
    result = predictor.predict(demo_img)
    print(json.dumps(result, indent=2))
else:
    print("Run streamlit run app/app.py for interactive browser demo!")